In [1]:
from database.rds_crypto_market_db_util import RdsCryptoMarketDbUtil
import pandas as pd
from storage.s3_storage import S3Storage
from config.config import AWSConf
from models.coins import Coins

In [2]:
def get_existing_coin_ids(rds_crypto_market_db_util):
    query = "SELECT api_coin_id FROM coins"
    result_set = rds_crypto_market_db_util.fetchall(query)
    return result_set

In [ ]:
rds_crypto_market_db_util = RdsCryptoMarketDbUtil()

storage = S3Storage(
            bucket_name=AWSConf.S3_RAW_STORAGE_BUCKET_NAME,
            profile_name=AWSConf.AWS_PROFILE_NAME,
        )

recent_object_by_hour = storage.get_latest_object_for_hour(year=2026, month=8, day=20, hour=11)

data_obj = storage.get_json_object(object_key=recent_object_by_hour['Key'])
snapshot_datetime = data_obj['snapshot_datetime']
sn_df = pd.DataFrame(data_obj['data'])

report_df = sn_df[['id', 'symbol', 'name', 'image']]
report_df = report_df.drop_duplicates(ignore_index=True)

report_df = report_df.rename(columns={
    "id": "api_coin_id",
    "image": "image_url"
})

existing_coin_ids = get_existing_coin_ids(rds_crypto_market_db_util)
existing_coin_ids = [i['api_coin_id'] for i in existing_coin_ids]

report_df = report_df[~report_df['api_coin_id'].isin(existing_coin_ids)]

if report_df.empty:
    pass
else:
    table_cols = ['coin_id', 'api_coin_id', 'symbol', 'name', 'image_url']

    record_list = list(report_df.itertuples(index=False, name=None))
    Coins.insert_coins_data(record_list, rds_crypto_market_db_util)

True

True